# Project Delphi

## 04 - Streamlit App

### Overview:

In this final notebook, we’ll turn our two trained XGBoost models into a simple, shareable web app using Streamlit.

**The goal**: let creators test *what-if* scenarios -- title, length, and timing -- and instantly forecast predicted **views** and **subscriber change**.

This notebook marks the **deployment** stage of Project Delphi, bringing the predictive models to life through an intuitive interface.

---

### The Plan
1. **Set up Environment**: install and import Streamlit, confirm paths to model artifacts
2. **Load Artifacts**: bring in both trained models, scalers, schemas, and metadata
3. **Build Input Form**: title, episode length, day of week, and month  
4. **Generate Predictions**: embed title text, build feature vector, and run both models
5. **Display Results**: show predicted lifetime views and subscriber delta  
6. **Polish UI**: add header (“MERLIN”), tagline, and clean layout for usability

---

### Why this matters
This step transforms our research pipeline into **Merlin** -- a **what-if engine** that lets creators test their ideas *before* publishing.

In [ ]:
from google.colab import files

files.download('/content/drive/MyDrive/Colab Notebooks/project_delphi/models/xgb_views_model.pkl')
files.download('/content/drive/MyDrive/Colab Notebooks/project_delphi/models/xgb_subs_model.pkl')
files.download('/content/drive/MyDrive/Colab Notebooks/project_delphi/models/xgb_subs_target_scaler.pkl')
files.download('/content/drive/MyDrive/Colab Notebooks/project_delphi/models/features_views.json')
files.download('/content/drive/MyDrive/Colab Notebooks/project_delphi/models/features_subs.json')

## Set up the environment

In [ ]:
! pip install streamlit

In [ ]:
# Import Streamlit and check the version
import streamlit as st
print("Streamlit version:", st.__version__)

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

## Load the artifacts

Now, let's **load and verify the model artifacts**. Before building the Streamlit interface, we'll confirm that all saved model artifacts (pickles, schemas, and metadata) can be accessed and read from Drive.

This way, we make sure the deployment environment matches the training setup from Notebooks 03a and 03b.

In [ ]:
from pathlib import Path
import joblib, json

MODELS_DIR = Path("/content/drive/MyDrive/Colab Notebooks/project_delphi/models")

# List what's there
print([p.name for p in MODELS_DIR.iterdir()])

# Define paths
VIEWS_MODEL = MODELS_DIR / "xgb_views_model.pkl"
SUBS_MODEL  = MODELS_DIR / "xgb_subs_model.pkl"
SUBS_SCALER = MODELS_DIR / "xgb_subs_target_scaler.pkl"
VIEWS_FEATS = MODELS_DIR / "features_views.json"
SUBS_FEATS  = MODELS_DIR / "features_subs.json"
VIEWS_META  = MODELS_DIR / "views_model_meta.json"
SUBS_META   = MODELS_DIR / "subs_model_meta.json"

# Quick existence check
for p in [VIEWS_MODEL, SUBS_MODEL, SUBS_SCALER, VIEWS_FEATS, SUBS_FEATS, VIEWS_META, SUBS_META]:
    print(p.name, "->", p.exists())

# Load one of each to confirm
views_model = joblib.load(VIEWS_MODEL)
with open(VIEWS_FEATS) as f: views_schema = json.load(f)["feature_cols"]
with open(VIEWS_META)  as f: views_meta   = json.load(f)

print("Loaded:", type(views_model).__name__, "| features:", len(views_schema), "| meta keys:", list(views_meta.keys()))

## Build the input form

Next, we'll create the user inputs that feed into Merlin's prediction engine. These inputs -- **title, episode length, day of week and month** -- will later become the interactive fields in the Streamlit app, driving how each scenario is evaluated.

In [ ]:
# Load the SentenceTransformer
# Note: The modl turns YouTube titles into 384-dimensional embeddings,
# capturing tone, phrsing and semantic meaning
!pip install -q sentence-transformers
from sentence_transformers import SentenceTransformer
embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

# Define examle inputs (to be replaced by Streamlit widget)
# We'll simulate onr "what-if" scenario
title = "How the Warriors Pulled Off a Miracle Comeback"
duration_seconds = 12 * 60      # Episode length in seconds
day_of_week = 3                  # Mon=0 .. Sun=6
month = 10                       # Jan=0 .. Dec=11

# Embed the title and assemble the feature dictionary
# Combine duration, day, and month with the 384-dimensional title vector into one unified input
import numpy as np
vec = embedder.encode([title], normalize_embeddings=True)[0]
feat_dict = {
    "duration_seconds": float(duration_seconds),
    "day_of_week": int(day_of_week),
    "month": int(month),
    **{f"embed_{i}": float(vec[i]) for i in range(384)}
}

# Re-order the features to match the training schemas
# Load the saved feature lists from trainin so that the columns line up perfectly
# Note: This guarantees the model sees the inputs in the exact same order and format
with open(VIEWS_FEATS) as f: views_schema = json.load(f)["feature_cols"]
with open(SUBS_FEATS)  as f: subs_schema  = json.load(f)["feature_cols"]

# Now, buidl the feature arrays for each of the two models
# Note: Any feature missing in feat_dict is safely filled with 0.0
X_views = np.array([[feat_dict.get(c, 0.0) for c in views_schema]])
X_subs  = np.array([[feat_dict.get(c, 0.0) for c in subs_schema]])

# Check the shapes of both arrays; Should both be (1,387)
X_views.shape, X_subs.shape

## Generate the predictions

Now that we’ve built the feature arrays, it’s time to **run both models** to see what Merlin predicts. The views model outputs log-transformed values, and the subs model outputs scaled values. So, we'll turn both back to their real-world numbers.

In [ ]:
# Load the models -- and the scaler for subs
views_model = joblib.load(VIEWS_MODEL)
subs_model = joblib.load(SUBS_MODEL)
subs_scaler = joblib.load(SUBS_SCALER)

# Predict lifetime views
# The views model was trained on log-trasnformed data, so we
# reverse that transform using np.exmp1() to get actual view counts
y_views_log = views_model.predict(X_views)
y_views = np.expm1(y_views_log).ravel()

# Now, to predict subscribers added (or lost)
# the subs model was trained on scaled datas, so we use the saved scaler to turn the
# predictions back to real subscriber changes
y_subs_scaled = subs_model.predict(X_subs)
y_subs = subs_scaler.inverse_transform(y_subs_scaled.reshape(-1, 1)).ravel()

# As a quick check, print both predictions -- and round for readibility
int(max(0, round(y_views[0]))), round(float(y_subs[0]))

## Display the results

Now that Merlin has made its predictions, let's show them in a clean, readable format. These will eventally become the results displayed inside the Streamlit app.

In [ ]:
# Turn the raw model outpurs into clean, readable numbers
# Note: The mkdel predicts arrays, so we take the first value [0]
# Also: We round to the nearest whole number since views ans subs aren't fractuional

# Use max(0...) to makse sure there are no negative predictions for views
pred_views = int(max(0, round(y_views[0])))

# Turn the flaots into integers -- so as not to have decimals
pred_subs = round(float(y_subs[0]))

# Now, print the results in a friendly format
# Note: Add commas for large view counts and a plus sign for positive subscriber gains
print("📈 Predicted Lifetime Views:", f"{pred_views:,}")
print("👥 Predicted Subscriber Change:", f"{'+' if pred_subs > 0 else ''}{pred_subs}")

## Polish the UI

Let’s connect everything together -- adding input fields, a predict button, and simple metric displays to tunr Merlin into a real app.



In [ ]:
# Save the Streamlit app file
# Note: This cell writes the Streamlit code below into
# and "app.py" so it can run as a standalong web app
# After running, launch it with: !streamlit run app.py
%%writefile app.py

# Start by importing Streamlit
import streamlit as st

# Set the title and tag line
st.title("MERLIN")
st.caption("The what-if engine for YouTube creators")

# Set the input buttons
title = st.text_input("Video Title")
duration_seconds = st.number_input("Episode Length (minutes)", min_value=1, step=1) * 60
day_of_week = st.selectbox("Day of Week", list(range(7)))
month = st.selectbox("Month", list(range(12)))

# When the user clicks "Predict", run the model and show the results
if st.button("Predict"):
    # (call your existing prediction logic here)
    st.metric("Predicted Lifetime Views", f"{pred_views:,}")
    st.metric("Predicted Subscriber Change", f"{'+' if pred_subs > 0 else ''}{pred_subs}")

In [ ]:
!ls -l


In [ ]:
!find / -name "app.py" 2>/dev/null


In [ ]:
from google.colab import files; files.download('app.py')
